In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

env_dir = Path.cwd()
for parent in [env_dir, *env_dir.parents]:
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        break
else:
    load_dotenv()

host = (os.getenv("DB_HOST") or "").strip().strip("'\"")
url = URL.create(
    "mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=host,
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)
engine = create_engine(
    url,
    connect_args={"ssl": {"ssl_mode": "REQUIRED"}, "charset": "utf8mb4"},
)

VIDEO_CSV = Path(
    r"C:/Users/user/DATA_MKT_260920/data.project/KDT_Integral_Data_Project/02_csv/01_youtube/1. videos/02. cleaning/02. 3NF_&_칼럼명_통합/youtube_analysis_video_table.csv"
)

create_sql = """
CREATE TABLE IF NOT EXISTS youtube_post (
    post_id VARCHAR(16) NOT NULL,
    account_id VARCHAR(32) NOT NULL,
    created_at DATETIME NOT NULL,
    post_title VARCHAR(255) NOT NULL,
    description TEXT NULL,
    view_count BIGINT NOT NULL,
    like_count BIGINT NOT NULL,
    comment_count BIGINT NOT NULL,
    collected_at DATE NOT NULL,
    PRIMARY KEY (post_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
"""

def load_video_table():
    df = pd.read_csv(VIDEO_CSV, encoding="utf-8-sig")
    df["created_at"] = (
        pd.to_datetime(df["created_at"], utc=True)
        .dt.tz_convert("Asia/Seoul")
        .dt.tz_localize(None)
    )
    df["collected_at"] = pd.to_datetime(df["collected_at"]).dt.date
    for col in ("view_count", "like_count", "comment_count"):
        df[col] = df[col].astype("int64")
    return df

with engine.begin() as conn:
    conn.execute(text(create_sql))
    has_column = conn.execute(text("""
        SELECT COUNT(*)
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = DATABASE()
          AND TABLE_NAME = 'youtube_post'
          AND COLUMN_NAME = 'collected_at'
    """)).scalar()
    if not has_column:
        conn.execute(text("ALTER TABLE youtube_post ADD COLUMN collected_at DATE NULL"))
    existing = conn.execute(text("SELECT COUNT(*) FROM youtube_post")).scalar()
    missing_collected = conn.execute(
        text("SELECT COUNT(*) FROM youtube_post WHERE collected_at IS NULL")
    ).scalar()

if existing and not missing_collected:
    print(f"이미 적재됨: {existing:,}행")
elif existing:
    df = load_video_table()
    rows = [
        {"post_id": post_id, "collected_at": collected_at}
        for post_id, collected_at in df[["post_id", "collected_at"]].itertuples(index=False, name=None)
    ]
    insert_collected = text(
        "INSERT INTO tmp_post_collected (post_id, collected_at) VALUES (:post_id, :collected_at)"
    )
    with engine.begin() as conn:
        conn.execute(text(
            "CREATE TEMPORARY TABLE tmp_post_collected ("
            "post_id VARCHAR(16) PRIMARY KEY, collected_at DATE NOT NULL)"
        ))
        for start in range(0, len(rows), 500):
            conn.execute(insert_collected, rows[start:start + 500])
        updated = conn.execute(text("""
            UPDATE youtube_post AS p
            JOIN tmp_post_collected AS t ON p.post_id = t.post_id
            SET p.collected_at = t.collected_at
        """)).rowcount
        still_null = conn.execute(
            text("SELECT COUNT(*) FROM youtube_post WHERE collected_at IS NULL")
        ).scalar()
        if still_null:
            raise RuntimeError(f"collected_at이 비어 있는 행이 {still_null}건 남았습니다.")
        conn.execute(text("ALTER TABLE youtube_post MODIFY collected_at DATE NOT NULL"))
    print(f"collected_at 반영: {updated:,}행")
else:
    df = load_video_table()
    df.to_sql(
        name="youtube_post",
        con=engine,
        if_exists="append",
        index=False,
        chunksize=500,
        method="multi",
    )
    print(f"테이블 생성 및 데이터 적재 완료: {len(df):,}행")

with engine.connect() as conn:
    collected_summary = pd.read_sql(
        text("SELECT collected_at, COUNT(*) AS n FROM youtube_post GROUP BY collected_at ORDER BY collected_at"),
        conn,
    )
print(collected_summary.to_string(index=False))

account_sql = """
CREATE TABLE IF NOT EXISTS youtube_account (
    account_id VARCHAR(32) NOT NULL,
    account_name VARCHAR(255) NOT NULL,
    PRIMARY KEY (account_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
"""

with engine.begin() as conn:
    conn.execute(text(account_sql))
    existing_accounts = conn.execute(text("SELECT COUNT(*) FROM youtube_account")).scalar()

if existing_accounts:
    print(f"youtube_account 이미 적재됨: {existing_accounts:,}행")
else:
    account_df = pd.read_csv("youtube_analysis_account_table.csv", encoding="utf-8-sig")
    account_df.to_sql(
        name="youtube_account",
        con=engine,
        if_exists="append",
        index=False,
        chunksize=500,
        method="multi",
    )
    print(f"youtube_account 적재 완료: {len(account_df):,}행")


collected_at 반영: 13,383행


collected_at    n
  2026-10-26 5894
  2026-10-27 7489


youtube_account 이미 적재됨: 5,668행
